# RT Notebook 20
## Distinction Conservation and Exclusion Dynamics

### Theory Series II — Primitive Mechanism Test

Notebook 18 tested whether projection necessarily generates residue.

Notebook 19 tested whether measured structural exclusion is equivalent to residue.

Notebook 20 replaces structural exclusion proxies with a direct operational definition:

\[
\boxed{
\text{Exclusion} := \text{loss of distinction}
}
\]

Projection is treated as reorganization or consumption of admissible distinction.

The central question is therefore not whether a mapping has collisions or omitted edges, but:

> Which distinctions present before projection remain distinguishable afterward?

# Primary definitions

Let a finite organization be represented by a state space \(\Omega\).

For any two distinct states \(a,b\in\Omega\), define a distinction token:

\[
d(a,b)
\]

A distinction is **preserved** by projection \(\Pi\) when:

\[
\Pi(a)\neq \Pi(b)
\]

A distinction is **lost** when:

\[
\Pi(a)=\Pi(b)
\]

The excluded distinction set is therefore:

\[
X_\Pi =
\left\{
d(a,b)\mid a\neq b,\ \Pi(a)=\Pi(b)
\right\}
\]

The preserved distinction set is:

\[
P_\Pi =
\left\{
d(a,b)\mid a\neq b,\ \Pi(a)\neq\Pi(b)
\right\}
\]

By construction:

\[
D_{\text{before}}
=
P_\Pi \,\dot{\cup}\, X_\Pi
\]

where \(\dot{\cup}\) denotes disjoint union.

# Primary hypothesis

## Distinction conservation

\[
H_1:
\quad
D_{\text{before}}
=
P_\Pi \,\dot{\cup}\, X_\Pi
\]

Equivalently:

\[
|D_{\text{before}}|
=
|P_\Pi|+|X_\Pi|
\]

This is not assumed merely because of bookkeeping. The notebook verifies the identity across the entire tested projection class and records any computational or definitional violations.

## Secondary hypothesis

Residue is determined by excluded distinction:

\[
H_2:
\quad
R=f(X)
\]

The notebook tests several candidate relationships:

\[
R=|X|
\]

\[
R\propto |X|
\]

\[
R=f(|X|,\text{type}(X))
\]

The form of \(f\) is not assumed in advance.

# Falsification conditions

The distinction-conservation hypothesis fails within the tested model class if any projection produces:

1. an original distinction classified as neither preserved nor excluded;
2. a distinction classified as both preserved and excluded;
3. a count mismatch:

\[
|D_{\text{before}}|
\neq
|P_\Pi|+|X_\Pi|
\]

The residue hypothesis fails in its strongest form if two projections with identical excluded distinction sets produce different residue values.

# Deliverables

Running this notebook produces:

- `projection_records.csv`
- `distinction_inventory.csv`
- `conservation_test_results.csv`
- `conservation_violations.csv`
- `residue_model_comparison.csv`
- `matched_exclusion_counterexamples.csv`
- `residue_by_exclusion_count.csv`
- `residue_by_exclusion_signature.csv`
- `block_manifest.csv`
- `figure_residue_vs_excluded_distinction.png`
- `figure_preserved_vs_excluded.png`
- `figure_residue_model_fit.png`
- `findings20.json`
- `run_manifest20.json`
- `RT_Notebook_20_outputs.zip`

## 1. Imports and run configuration

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations, product
from pathlib import Path
from typing import Iterable, Optional, Tuple, List, Dict
from collections import Counter, defaultdict
import hashlib
import json
import math
import os
import platform
import random
import statistics
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 200020
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("outputs_notebook20")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "domains": ["binary", "ternary", "relational"],
    "source_sizes": [2, 3, 4],
    "target_sizes": [1, 2, 3, 4],
    "max_source_states_per_block": 96,
    "max_target_states_per_block": 96,
    "seed": SEED,
}

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output directory:", OUTPUT_DIR.resolve())
print(json.dumps(CONFIG, indent=2))

## 2. Finite relational organizations

In [ ]:
@dataclass(frozen=True)
class Organization:
    domain: str
    labels: Tuple[int, ...]
    edges: Tuple[Tuple[int, int], ...]
    reference: Optional[int]

    @property
    def size(self) -> int:
        return len(self.labels)

    def uid(self) -> str:
        payload = {
            "domain": self.domain,
            "labels": self.labels,
            "edges": self.edges,
            "reference": self.reference,
        }
        raw = json.dumps(payload, sort_keys=True).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()[:16]


DOMAIN_LABELS = {
    "binary": (-1, 1),
    "ternary": (-1, 0, 1),
    "relational": (-1, 0, 1),
}


def all_edges(n: int) -> Tuple[Tuple[int, int], ...]:
    return tuple(combinations(range(n), 2))


def powerset_edges(n: int) -> Iterable[Tuple[Tuple[int, int], ...]]:
    candidates = all_edges(n)
    for mask in range(1 << len(candidates)):
        yield tuple(
            candidates[i]
            for i in range(len(candidates))
            if mask & (1 << i)
        )


def generate_organizations(domain: str, n: int) -> Iterable[Organization]:
    label_space = product(DOMAIN_LABELS[domain], repeat=n)

    if domain == "relational":
        edge_space = list(powerset_edges(n))
        reference_space = list(range(n))
    else:
        edge_space = [tuple()]
        reference_space = [None]

    for labels in label_space:
        for edges in edge_space:
            for reference in reference_space:
                yield Organization(
                    domain=domain,
                    labels=tuple(labels),
                    edges=tuple(sorted(edges)),
                    reference=reference,
                )


def deterministic_subset(items: List[Organization], limit: int) -> List[Organization]:
    if len(items) <= limit:
        return items
    ordered = sorted(items, key=lambda x: x.uid())
    step = len(ordered) / limit
    indices = sorted({
        min(len(ordered) - 1, int(i * step))
        for i in range(limit)
    })
    return [ordered[i] for i in indices]


for domain in CONFIG["domains"]:
    for n in CONFIG["source_sizes"]:
        count = sum(1 for _ in generate_organizations(domain, n))
        print(f"{domain:10s} n={n}: {count:,}")

## 3. State-level distinction inventory

In [ ]:
@dataclass(frozen=True)
class Distinction:
    left: int
    right: int
    label_difference: bool
    relation_profile_difference: bool
    reference_difference: bool

    def token(self) -> str:
        return json.dumps({
            "pair": [self.left, self.right],
            "label_difference": self.label_difference,
            "relation_profile_difference": self.relation_profile_difference,
            "reference_difference": self.reference_difference,
        }, sort_keys=True)


def relation_profile(org: Organization, node: int) -> Tuple[int, ...]:
    neighbors = []
    edge_set = set(org.edges)
    for other in range(org.size):
        if other == node:
            continue
        edge = tuple(sorted((node, other)))
        neighbors.append(int(edge in edge_set))
    return tuple(neighbors)


def distinction_inventory(org: Organization) -> List[Distinction]:
    inventory = []

    for a, b in combinations(range(org.size), 2):
        label_difference = org.labels[a] != org.labels[b]
        relation_difference = relation_profile(org, a) != relation_profile(org, b)
        reference_difference = (
            org.reference is not None
            and ((org.reference == a) != (org.reference == b))
        )

        # Every pair of distinct members is itself a primitive distinction.
        # The three booleans record typed aspects of that distinction.
        inventory.append(
            Distinction(
                left=a,
                right=b,
                label_difference=label_difference,
                relation_profile_difference=relation_difference,
                reference_difference=reference_difference,
            )
        )

    return inventory


sample = Organization(
    domain="relational",
    labels=(-1, 0, 1),
    edges=((0, 1),),
    reference=0,
)

for distinction in distinction_inventory(sample):
    print(distinction)

## 4. Projection and direct exclusion definition

In [ ]:
def all_total_maps(n_source: int, n_target: int):
    return product(range(n_target), repeat=n_source)


def projected_distinction_status(
    distinction: Distinction,
    mapping: Tuple[int, ...],
) -> str:
    return (
        "excluded"
        if mapping[distinction.left] == mapping[distinction.right]
        else "preserved"
    )


def excluded_distinctions(
    org: Organization,
    mapping: Tuple[int, ...],
) -> List[Distinction]:
    return [
        d
        for d in distinction_inventory(org)
        if projected_distinction_status(d, mapping) == "excluded"
    ]


def preserved_distinctions(
    org: Organization,
    mapping: Tuple[int, ...],
) -> List[Distinction]:
    return [
        d
        for d in distinction_inventory(org)
        if projected_distinction_status(d, mapping) == "preserved"
    ]


def exclusion_signature(
    org: Organization,
    mapping: Tuple[int, ...],
) -> str:
    tokens = sorted(d.token() for d in excluded_distinctions(org, mapping))
    return hashlib.sha256(
        json.dumps(tokens).encode("utf-8")
    ).hexdigest()[:16]

## 5. Target realization and residue

In [ ]:
def project_labels(
    source: Organization,
    mapping: Tuple[int, ...],
    target_size: int,
) -> Tuple[int, ...]:
    buckets = [[] for _ in range(target_size)]

    for source_index, target_index in enumerate(mapping):
        buckets[target_index].append(source.labels[source_index])

    result = []
    for values in buckets:
        if not values:
            result.append(0)
        elif all(v == values[0] for v in values):
            result.append(values[0])
        else:
            # Incompatible distinctions resolve to an undecided local value.
            result.append(0)

    return tuple(result)


def project_edges(
    source: Organization,
    mapping: Tuple[int, ...],
) -> Tuple[Tuple[int, int], ...]:
    image_edges = set()

    for a, b in source.edges:
        x, y = mapping[a], mapping[b]
        if x != y:
            image_edges.add(tuple(sorted((x, y))))

    return tuple(sorted(image_edges))


def projected_reference(
    source: Organization,
    mapping: Tuple[int, ...],
) -> Optional[int]:
    if source.reference is None:
        return None
    return mapping[source.reference]


@dataclass(frozen=True)
class Residue:
    label_mismatch: int
    relation_mismatch: int
    reference_mismatch: int
    inverse_failure: int

    @property
    def total(self) -> int:
        return (
            self.label_mismatch
            + self.relation_mismatch
            + self.reference_mismatch
            + self.inverse_failure
        )


def compute_residue(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> Residue:
    expected_labels = project_labels(source, mapping, target.size)
    expected_edges = set(project_edges(source, mapping))
    expected_reference = projected_reference(source, mapping)

    label_mismatch = sum(
        int(a != b)
        for a, b in zip(expected_labels, target.labels)
    )

    relation_mismatch = len(
        expected_edges.symmetric_difference(set(target.edges))
    )

    reference_mismatch = int(
        expected_reference != target.reference
    )

    inverse_failure = int(
        not (
            source.size == target.size
            and len(set(mapping)) == source.size
        )
    )

    return Residue(
        label_mismatch=label_mismatch,
        relation_mismatch=relation_mismatch,
        reference_mismatch=reference_mismatch,
        inverse_failure=inverse_failure,
    )

## 6. Campaign construction

In [ ]:
def campaign_blocks():
    for source_domain in CONFIG["domains"]:
        for target_domain in CONFIG["domains"]:
            if source_domain == target_domain:
                continue

            for source_size in CONFIG["source_sizes"]:
                for target_size in CONFIG["target_sizes"]:
                    source_orgs_all = list(
                        generate_organizations(source_domain, source_size)
                    )
                    target_orgs_all = list(
                        generate_organizations(target_domain, target_size)
                    )

                    source_orgs = deterministic_subset(
                        source_orgs_all,
                        CONFIG["max_source_states_per_block"],
                    )
                    target_orgs = deterministic_subset(
                        target_orgs_all,
                        CONFIG["max_target_states_per_block"],
                    )

                    yield {
                        "source_domain": source_domain,
                        "target_domain": target_domain,
                        "source_size": source_size,
                        "target_size": target_size,
                        "source_orgs": source_orgs,
                        "target_orgs": target_orgs,
                        "source_total": len(source_orgs_all),
                        "target_total": len(target_orgs_all),
                        "source_used": len(source_orgs),
                        "target_used": len(target_orgs),
                        "mode": (
                            "exhaustive"
                            if (
                                len(source_orgs) == len(source_orgs_all)
                                and len(target_orgs) == len(target_orgs_all)
                            )
                            else "deterministic_stratified"
                        ),
                    }

## 7. Execute experiment

In [ ]:
projection_rows = []
inventory_rows = []
block_manifest = []

start = time.time()

for block_index, block in enumerate(campaign_blocks(), start=1):
    local_records = 0

    for source in block["source_orgs"]:
        source_inventory = distinction_inventory(source)

        for target in block["target_orgs"]:
            for mapping in all_total_maps(source.size, target.size):
                preserved = preserved_distinctions(source, mapping)
                excluded = excluded_distinctions(source, mapping)
                residue = compute_residue(source, target, mapping)

                before_tokens = {d.token() for d in source_inventory}
                preserved_tokens = {d.token() for d in preserved}
                excluded_tokens = {d.token() for d in excluded}

                overlap = preserved_tokens & excluded_tokens
                union = preserved_tokens | excluded_tokens

                conservation_holds = (
                    len(overlap) == 0
                    and union == before_tokens
                    and len(before_tokens)
                    == len(preserved_tokens) + len(excluded_tokens)
                )

                label_excluded = sum(d.label_difference for d in excluded)
                relation_excluded = sum(
                    d.relation_profile_difference for d in excluded
                )
                reference_excluded = sum(
                    d.reference_difference for d in excluded
                )

                projection_rows.append({
                    "source_domain": source.domain,
                    "target_domain": target.domain,
                    "source_size": source.size,
                    "target_size": target.size,
                    "source_uid": source.uid(),
                    "target_uid": target.uid(),
                    "mapping": json.dumps(mapping),

                    "distinctions_before": len(before_tokens),
                    "distinctions_preserved": len(preserved_tokens),
                    "distinctions_excluded": len(excluded_tokens),
                    "label_distinctions_excluded": label_excluded,
                    "relation_distinctions_excluded": relation_excluded,
                    "reference_distinctions_excluded": reference_excluded,
                    "exclusion_signature": exclusion_signature(source, mapping),

                    "preserved_excluded_overlap": len(overlap),
                    "classified_union_size": len(union),
                    "conservation_holds": conservation_holds,

                    "label_residue": residue.label_mismatch,
                    "relation_residue": residue.relation_mismatch,
                    "reference_residue": residue.reference_mismatch,
                    "inverse_residue": residue.inverse_failure,
                    "total_residue": residue.total,
                })

                for distinction in source_inventory:
                    status = projected_distinction_status(distinction, mapping)
                    inventory_rows.append({
                        "source_uid": source.uid(),
                        "target_uid": target.uid(),
                        "mapping": json.dumps(mapping),
                        "distinction_token": distinction.token(),
                        "status": status,
                        "label_difference": distinction.label_difference,
                        "relation_profile_difference": (
                            distinction.relation_profile_difference
                        ),
                        "reference_difference": distinction.reference_difference,
                    })

                local_records += 1

    block_manifest.append({
        "block_index": block_index,
        "source_domain": block["source_domain"],
        "target_domain": block["target_domain"],
        "source_size": block["source_size"],
        "target_size": block["target_size"],
        "source_organizations_total": block["source_total"],
        "target_organizations_total": block["target_total"],
        "source_organizations_used": block["source_used"],
        "target_organizations_used": block["target_used"],
        "mode": block["mode"],
        "projection_records": local_records,
    })

elapsed = time.time() - start

projection_df = pd.DataFrame(projection_rows)
inventory_df = pd.DataFrame(inventory_rows)
block_manifest_df = pd.DataFrame(block_manifest)

print(f"Projection records: {len(projection_df):,}")
print(f"Distinction records: {len(inventory_df):,}")
print(f"Elapsed seconds: {elapsed:,.2f}")

## 8. Conservation validation

In [ ]:
conservation_summary = pd.DataFrame([
    {
        "tested_projections": len(projection_df),
        "conservation_passes": int(
            projection_df["conservation_holds"].sum()
        ),
        "conservation_failures": int(
            (~projection_df["conservation_holds"]).sum()
        ),
        "pass_rate": float(
            projection_df["conservation_holds"].mean()
        ),
        "overlap_violations": int(
            (projection_df["preserved_excluded_overlap"] > 0).sum()
        ),
        "count_violations": int(
            (
                projection_df["distinctions_before"]
                != projection_df["distinctions_preserved"]
                + projection_df["distinctions_excluded"]
            ).sum()
        ),
    }
])

conservation_violations = projection_df[
    ~projection_df["conservation_holds"]
].copy()

conservation_summary

## 9. Residue model comparison

In [ ]:
def linear_fit_metrics(x: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    if len(np.unique(x)) < 2:
        return {
            "slope": float("nan"),
            "intercept": float(np.mean(y)),
            "r_squared": float("nan"),
            "mae": float(np.mean(np.abs(y - np.mean(y)))),
        }

    slope, intercept = np.polyfit(x, y, 1)
    prediction = slope * x + intercept

    ss_res = float(np.sum((y - prediction) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    mae = float(np.mean(np.abs(y - prediction)))

    return {
        "slope": float(slope),
        "intercept": float(intercept),
        "r_squared": r_squared,
        "mae": mae,
    }


model_rows = []

targets = [
    "total_residue",
    "label_residue",
    "relation_residue",
    "reference_residue",
    "inverse_residue",
]

predictors = [
    "distinctions_excluded",
    "label_distinctions_excluded",
    "relation_distinctions_excluded",
    "reference_distinctions_excluded",
]

for target in targets:
    y = projection_df[target].to_numpy(dtype=float)

    for predictor in predictors:
        x = projection_df[predictor].to_numpy(dtype=float)
        metrics = linear_fit_metrics(x, y)

        model_rows.append({
            "target": target,
            "predictor": predictor,
            **metrics,
            "pearson_r": float(
                np.corrcoef(x, y)[0, 1]
            ) if np.std(x) > 0 and np.std(y) > 0 else float("nan"),
        })

residue_model_comparison = pd.DataFrame(model_rows)
residue_model_comparison.sort_values(
    ["target", "r_squared"],
    ascending=[True, False],
)

## 10. Strong-form test: identical exclusion, different residue

In [ ]:
signature_groups = (
    projection_df
    .groupby("exclusion_signature")
    .agg(
        projection_count=("exclusion_signature", "size"),
        excluded_count=("distinctions_excluded", "first"),
        residue_min=("total_residue", "min"),
        residue_max=("total_residue", "max"),
        residue_nunique=("total_residue", "nunique"),
        source_domain_nunique=("source_domain", "nunique"),
        target_domain_nunique=("target_domain", "nunique"),
    )
    .reset_index()
)

matched_exclusion_counterexamples = signature_groups[
    signature_groups["residue_nunique"] > 1
].copy()

strong_form_survives = len(matched_exclusion_counterexamples) == 0

print("Unique exclusion signatures:", len(signature_groups))
print(
    "Identical exclusion signatures with differing residue:",
    len(matched_exclusion_counterexamples),
)
print(
    "Strong form R=f(X):",
    "NOT_FALSIFIED" if strong_form_survives else "FALSIFIED",
)

## 11. Residue summaries

In [ ]:
residue_by_exclusion_count = (
    projection_df
    .groupby("distinctions_excluded")
    .agg(
        projection_count=("distinctions_excluded", "size"),
        mean_total_residue=("total_residue", "mean"),
        median_total_residue=("total_residue", "median"),
        minimum_total_residue=("total_residue", "min"),
        maximum_total_residue=("total_residue", "max"),
        mean_label_residue=("label_residue", "mean"),
        mean_relation_residue=("relation_residue", "mean"),
        mean_reference_residue=("reference_residue", "mean"),
        mean_inverse_residue=("inverse_residue", "mean"),
    )
    .reset_index()
)

residue_by_exclusion_signature = (
    projection_df
    .groupby("exclusion_signature")
    .agg(
        projection_count=("exclusion_signature", "size"),
        distinctions_excluded=("distinctions_excluded", "first"),
        mean_total_residue=("total_residue", "mean"),
        residue_variance=("total_residue", "var"),
        residue_min=("total_residue", "min"),
        residue_max=("total_residue", "max"),
    )
    .reset_index()
)

residue_by_exclusion_count

## 12. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    projection_df["distinctions_excluded"],
    projection_df["total_residue"],
    alpha=0.2,
)
ax.set_xlabel("Excluded distinctions")
ax.set_ylabel("Total residue")
ax.set_title("Residue versus excluded distinction")
fig.tight_layout()

figure1 = OUTPUT_DIR / "figure_residue_vs_excluded_distinction.png"
fig.savefig(figure1, dpi=180)
plt.show()

In [ ]:
aggregate = (
    projection_df
    .groupby("distinctions_before")
    .agg(
        mean_preserved=("distinctions_preserved", "mean"),
        mean_excluded=("distinctions_excluded", "mean"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(
    aggregate["distinctions_before"],
    aggregate["mean_preserved"],
    marker="o",
    label="Preserved",
)
ax.plot(
    aggregate["distinctions_before"],
    aggregate["mean_excluded"],
    marker="o",
    label="Excluded",
)
ax.set_xlabel("Distinctions before projection")
ax.set_ylabel("Mean distinction count")
ax.set_title("Preserved and excluded distinction")
ax.legend()
fig.tight_layout()

figure2 = OUTPUT_DIR / "figure_preserved_vs_excluded.png"
fig.savefig(figure2, dpi=180)
plt.show()

In [ ]:
x = projection_df["distinctions_excluded"].to_numpy(dtype=float)
y = projection_df["total_residue"].to_numpy(dtype=float)

fit = linear_fit_metrics(x, y)
prediction = fit["slope"] * x + fit["intercept"]

order = np.argsort(x)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x, y, alpha=0.15)
ax.plot(x[order], prediction[order])
ax.set_xlabel("Excluded distinctions")
ax.set_ylabel("Total residue")
ax.set_title(
    "Linear candidate model "
    f"(R²={fit['r_squared']:.4f})"
)
fig.tight_layout()

figure3 = OUTPUT_DIR / "figure_residue_model_fit.png"
fig.savefig(figure3, dpi=180)
plt.show()

## 13. Interpretation protocol

In [ ]:
conservation_passed = (
    int(conservation_summary.iloc[0]["conservation_failures"]) == 0
)

if conservation_passed:
    conservation_verdict = "NOT_FALSIFIED_IN_TESTED_MODEL_CLASS"
    conservation_interpretation = (
        "Every source distinction was classified exactly once as preserved "
        "or excluded. Distinction conservation survived the tested model class."
    )
else:
    conservation_verdict = "FALSIFIED_IN_TESTED_MODEL_CLASS"
    conservation_interpretation = (
        "At least one projection failed the distinction partition identity."
    )

if strong_form_survives:
    strong_form_verdict = "NOT_FALSIFIED_IN_TESTED_MODEL_CLASS"
    strong_form_interpretation = (
        "No identical excluded-distinction set produced differing residue."
    )
else:
    strong_form_verdict = "FALSIFIED_IN_TESTED_MODEL_CLASS"
    strong_form_interpretation = (
        "At least one identical excluded-distinction set produced multiple "
        "residue values. Excluded distinction alone is therefore insufficient "
        "to determine residue under the present residue definition."
    )

print("DISTINCTION CONSERVATION")
print(conservation_verdict)
print(conservation_interpretation)
print()
print("STRONG FORM R=f(X)")
print(strong_form_verdict)
print(strong_form_interpretation)

## 14. Export findings

In [ ]:
projection_records_path = OUTPUT_DIR / "projection_records.csv"
distinction_inventory_path = OUTPUT_DIR / "distinction_inventory.csv"
conservation_results_path = OUTPUT_DIR / "conservation_test_results.csv"
conservation_violations_path = OUTPUT_DIR / "conservation_violations.csv"
residue_models_path = OUTPUT_DIR / "residue_model_comparison.csv"
matched_counterexamples_path = (
    OUTPUT_DIR / "matched_exclusion_counterexamples.csv"
)
residue_by_count_path = OUTPUT_DIR / "residue_by_exclusion_count.csv"
residue_by_signature_path = (
    OUTPUT_DIR / "residue_by_exclusion_signature.csv"
)
block_manifest_path = OUTPUT_DIR / "block_manifest.csv"

projection_df.to_csv(projection_records_path, index=False)
inventory_df.to_csv(distinction_inventory_path, index=False)
conservation_summary.to_csv(conservation_results_path, index=False)
conservation_violations.to_csv(conservation_violations_path, index=False)
residue_model_comparison.to_csv(residue_models_path, index=False)
matched_exclusion_counterexamples.to_csv(
    matched_counterexamples_path,
    index=False,
)
residue_by_exclusion_count.to_csv(residue_by_count_path, index=False)
residue_by_exclusion_signature.to_csv(
    residue_by_signature_path,
    index=False,
)
block_manifest_df.to_csv(block_manifest_path, index=False)

print("CSV deliverables written.")

In [ ]:
best_models = (
    residue_model_comparison
    .sort_values("r_squared", ascending=False)
    .head(10)
    .replace({np.nan: None})
    .to_dict(orient="records")
)

findings = {
    "notebook": 20,
    "title": "Distinction Conservation and Exclusion Dynamics",
    "primitive_definition": {
        "exclusion": "loss of distinction",
        "preservation": "distinction remains distinguishable after projection",
        "projection": "reorganization or consumption of admissible distinction",
    },
    "primary_hypothesis": (
        "Every source distinction is partitioned exactly once into preserved "
        "or excluded distinction."
    ),
    "secondary_hypothesis": (
        "Residue is determined by the excluded distinction set."
    ),
    "results": {
        "tested_projections": len(projection_df),
        "tested_distinction_instances": len(inventory_df),
        "conservation_verdict": conservation_verdict,
        "conservation_failures": int(
            conservation_summary.iloc[0]["conservation_failures"]
        ),
        "strong_form_residue_verdict": strong_form_verdict,
        "matched_exclusion_counterexample_count": len(
            matched_exclusion_counterexamples
        ),
        "best_linear_models": best_models,
    },
    "interpretation": {
        "conservation": conservation_interpretation,
        "strong_form": strong_form_interpretation,
    },
    "scope": {
        "domains": CONFIG["domains"],
        "source_sizes": CONFIG["source_sizes"],
        "target_sizes": CONFIG["target_sizes"],
        "sampling": (
            "exhaustive where block size permits; otherwise deterministic "
            "stratified subset"
        ),
    },
    "limitations": [
        "Finite bounded model class.",
        "Distinctions are pairwise state distinctions.",
        "Residue remains structurally operationalized.",
        "A surviving conservation identity may partly reflect the partition definition.",
        "The strong-form test depends on exact exclusion signatures within sampled blocks.",
    ],
    "recommended_next_notebook": {
        "notebook": 21,
        "title": "Typed Distinction Loss and Residue Coupling",
        "question": (
            "Which typed losses of distinction are sufficient for which "
            "typed residues?"
        ),
    },
}

findings_path = OUTPUT_DIR / "findings20.json"
findings_path.write_text(
    json.dumps(findings, indent=2),
    encoding="utf-8",
)

run_manifest = {
    "notebook": 20,
    "seed": SEED,
    "python": sys.version,
    "platform": platform.platform(),
    "elapsed_seconds": elapsed,
    "projection_record_count": len(projection_df),
    "distinction_record_count": len(inventory_df),
    "configuration": CONFIG,
    "blocks": block_manifest,
    "artifacts": [
        str(projection_records_path),
        str(distinction_inventory_path),
        str(conservation_results_path),
        str(conservation_violations_path),
        str(residue_models_path),
        str(matched_counterexamples_path),
        str(residue_by_count_path),
        str(residue_by_signature_path),
        str(block_manifest_path),
        str(figure1),
        str(figure2),
        str(figure3),
        str(findings_path),
    ],
}

manifest_path = OUTPUT_DIR / "run_manifest20.json"
manifest_path.write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(findings, indent=2))

## 15. Package outputs

In [ ]:
zip_path = OUTPUT_DIR / "RT_Notebook_20_outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path == zip_path:
            continue
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Archive:", zip_path.resolve())
print("Archive size:", zip_path.stat().st_size, "bytes")

# Reporting rule

The notebook must report two claims separately.

## Claim 1 — Distinction conservation

\[
D_{\text{before}}
=
P_\Pi \,\dot{\cup}\, X_\Pi
\]

This tests whether exclusion, defined as loss of distinction, forms a complete partition with preservation.

## Claim 2 — Residue determination

\[
R=f(X)
\]

This tests whether the excluded distinction set is sufficient to determine residue.

A result supporting Claim 1 but falsifying Claim 2 means:

> exclusion has been defined coherently, but residue depends on excluded distinction plus additional relational context.

That outcome is theoretically informative and must not be reported as a failure of the entire framework.